# Orang 5 - Testing, Eksperimen & Grafik
Menjalankan program dari `03_genetic_algorithm_implementation.ipynb` tanpa mendefinisikan ulang fungsi algoritma genetika.
Melakukan eksperimen minimal 3 konfigurasi parameter, mencatat hasil, membuat grafik konvergensi, file Excel, dan ringkasan.

In [ ]:
%%capture
# Menjalankan seluruh program dari 03 untuk mengimpor fungsi dan data yang diperlukan
# %%capture digunakan agar output panjang dari 03 tidak mengotori notebook 04
%run -i 03_genetic_algorithm_implementation.ipynb

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt

# Konfigurasi Eksperimen sesuai instruksi
# Ditambahkan 2 konfigurasi lagi agar hasil pengujian lebih komprehensif
configs = [
    {"Konfigurasi": 1, "Populasi": 30, "Generasi": 50, "Crossover Rate": 0.7, "Mutation Rate": 0.1},
    {"Konfigurasi": 2, "Populasi": 50, "Generasi": 100, "Crossover Rate": 0.8, "Mutation Rate": 0.1},
    {"Konfigurasi": 3, "Populasi": 100, "Generasi": 150, "Crossover Rate": 0.9, "Mutation Rate": 0.2},
    {"Konfigurasi": 4, "Populasi": 150, "Generasi": 200, "Crossover Rate": 0.85, "Mutation Rate": 0.15},
    {"Konfigurasi": 5, "Populasi": 80, "Generasi": 80, "Crossover Rate": 0.75, "Mutation Rate": 0.25}
]

hasil_eksperimen = []
history_all = []

print("Memulai pengujian konfigurasi parameter...")

for conf in configs:
    print(f"\nMenjalankan Konfigurasi {conf['Konfigurasi']}...")
    
    # Mengatur variabel global di dalam namespace fungsi dari 03 agar perubahannya berdampak
    run_genetic_algorithm.__globals__['POP_SIZE'] = conf["Populasi"]
    run_genetic_algorithm.__globals__['MAX_GEN'] = conf["Generasi"]
    run_genetic_algorithm.__globals__['P_C'] = conf["Crossover Rate"]
    run_genetic_algorithm.__globals__['P_M'] = conf["Mutation Rate"]
    
    # Mulai menghitung waktu
    start_time = time.time()
    
    # Memanggil algoritma genetika menggunakan data dari 03
    best_kromosom, best_fit, history = run_genetic_algorithm(data_estimasi, NUM_ORDERS)
    
    # Selesai komputasi
    waktu_komputasi = time.time() - start_time
    
    # Menghitung metrik eksperimen
    # Generasi terbaik = indeks dimana best_fitness pertama kali muncul di history (+1 karena gen mulai dari 1)
    generasi_terbaik = history.index(best_fit) + 1 if best_fit in history else conf["Generasi"]
    rata_rata_fitness = sum(history) / len(history) if history else 0
    
    # Menyimpan hasil (dibulatkan agar rapi)
    hasil_eksperimen.append({
        "Konfigurasi": conf["Konfigurasi"],
        "Populasi": conf["Populasi"],
        "Generasi": conf["Generasi"],
        "Crossover Rate": conf["Crossover Rate"],
        "Mutation Rate": conf["Mutation Rate"],
        "Best Fitness": round(best_fit, 6),
        "Rata-rata Fitness": round(rata_rata_fitness, 6),
        "Waktu Komputasi (s)": round(waktu_komputasi, 2),
        "Generasi Terbaik": generasi_terbaik
    })
    
    history_all.append({
        "Konfigurasi": conf["Konfigurasi"],
        "History": history
    })

# Tampilkan output tabel eksperimen
df_hasil = pd.DataFrame(hasil_eksperimen)
print("\n--- TABEL HASIL EKSPERIMEN ---")
display(df_hasil)


In [ ]:
# Membuat grafik konvergensi
plt.figure(figsize=(10, 6))
for hist in history_all:
    plt.plot(hist["History"], label=f"Konfigurasi {hist['Konfigurasi']}")
    
plt.title("Grafik Konvergensi Algoritma Genetika (Eksperimen)")
plt.xlabel("Generasi")
plt.ylabel("Best Fitness")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Membuat file Excel hasil eksperimen
excel_filename = "hasil_eksperimen_Orang5.xlsx"
df_hasil.to_excel(excel_filename, index=False)
print(f"File excel hasil eksperimen berhasil disimpan sebagai: {excel_filename}")


In [ ]:
# Menentukan parameter terbaik dan menulis ringkasan
best_idx = df_hasil["Best Fitness"].idxmax()
best_config = df_hasil.loc[best_idx]

summary = f"""
===========================================================
RINGKASAN HASIL EKSPERIMEN UNTUK ORANG 1
===========================================================

Berdasarkan pengujian terhadap {len(configs)} konfigurasi parameter yang berbeda, 
didapatkan bahwa Konfigurasi {best_config['Konfigurasi']} memberikan hasil terbaik.

Detail Parameter Terbaik:
- Populasi        : {best_config['Populasi']}
- Generasi        : {best_config['Generasi']}
- Crossover Rate  : {best_config['Crossover Rate']}
- Mutation Rate   : {best_config['Mutation Rate']}

Hasil yang Dicapai:
- Best Fitness       : {best_config['Best Fitness']}
- Rata-rata Fitness  : {best_config['Rata-rata Fitness']}
- Generasi Terbaik   : Ditemukan pada generasi ke-{best_config['Generasi Terbaik']}
- Waktu Komputasi    : {best_config['Waktu Komputasi (s)']} detik

Kesimpulan:
Konfigurasi dengan ukuran populasi {best_config['Populasi']} dan generasi {best_config['Generasi']} 
(Crossover Rate {best_config['Crossover Rate']}, Mutation Rate {best_config['Mutation Rate']}) 
mampu mencapai konvergensi dan menghasilkan nilai fitness paling optimal dibandingkan konfigurasi lainnya.
Waktu komputasi yang dibutuhkan adalah {best_config['Waktu Komputasi (s)']} detik, 
sehingga parameter ini direkomendasikan untuk digunakan dalam implementasi final sistem.
===========================================================
"""

print(summary)

# Simpan ringkasan ke file teks
with open("ringkasan_eksperimen.txt", "w") as f:
    f.write(summary)
print("Ringkasan telah disimpan ke dalam file 'ringkasan_eksperimen.txt'.")
